In [55]:
import warnings
import os
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import pandas as pd
import numpy as np
import random
import sc_toolbox

import rpy2.rinterface_lib.callbacks
import anndata2ri
import logging
from pathlib import Path
from rpy2.robjects import pandas2ri, numpy2ri
from rpy2.robjects import r
from scipy.sparse import csr_matrix
import anndata as ad
sc.settings.verbosity = 0
rpy2.rinterface_lib.callbacks.logger.setLevel(logging.ERROR)

# pandas2ri.activate()
# numpy2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [56]:
%%R
library(edgeR)
library(MAST)

In [57]:
project_path = Path(
    "/Users/nicolebussola/Desktop/Projects/LBP_MountSinai/LBP_brain_blood_pairs/data/TD005881/"
)
adata_brain = sc.read(project_path / "brain_filtered_normalized.h5ad")
adata_blood = sc.read(project_path / "blood_filtered_normalized.h5ad")

In [58]:
adata_brain.shape, adata_blood.shape

((42575, 31107), (123768, 29526))

In [59]:
blood_anno =  sc.read(project_path / "blood_consensus_annotated.h5ad").obs
brain_anno =  sc.read(project_path / "brain_consensus_annotated.h5ad").obs

In [60]:
adata_blood.obs = adata_blood.obs.merge(blood_anno[['label_scarches','cell']], left_index=True, right_index=True, how='outer')
adata_brain.obs = adata_brain.obs.merge(brain_anno['manual_celltype_annotation_bp_15'], left_index=True, right_index=True, how='outer')
adata_brain.shape, adata_blood.shape, adata_brain.obs.shape, adata_blood.obs.shape

((42575, 31107), (123768, 29526), (42575, 43), (123768, 44))

In [61]:
meta_df = pd.read_csv(project_path /"Batch2_celllevel_metadata.tsv",sep="\t")
meta_df['donor'] = meta_df['donor'].apply(lambda x: ('-').join((x.split('-')[0], x.split('-')[1][1:])))
age_dict = meta_df.groupby('donor')['age'].agg(lambda x: x.unique()[0]).to_dict()
sex_dict = meta_df.groupby('donor')['sex'].agg(lambda x: x.unique()[0]).to_dict()
diag_dict = meta_df.groupby('donor')['diagnosis'].agg(lambda x: x.unique()[0]).to_dict()
adata_blood.obs = adata_blood.obs[['side', 'pt', 'log1p_n_genes_by_counts','log1p_total_counts','pct_counts_in_top_20_genes', "pct_counts_mt", 'pct_counts_ribo', 'label_scarches']]
adata_blood.obs = adata_blood.obs.rename({'label_scarches':'cell_type'}, axis=1)
adata_blood.obs['age'] = adata_blood.obs['pt'].map(age_dict).astype(int)
adata_blood.obs['sex'] = adata_blood.obs['pt'].map(sex_dict)
adata_blood.obs['diagnosis'] = adata_blood.obs['pt'].map(diag_dict)
adata_blood.obs['tissue'] = 'blood'
adata_brain.obs = adata_brain.obs[['side', 'pt', 'log1p_n_genes_by_counts','log1p_total_counts','pct_counts_in_top_20_genes', "pct_counts_mt", 'pct_counts_ribo', 'manual_celltype_annotation_bp_15']]
adata_brain.obs = adata_brain.obs.rename({'manual_celltype_annotation_bp_15':'cell_type'}, axis=1)

adata_brain.obs['age'] = adata_brain.obs['pt'].map(age_dict).astype(int)
adata_brain.obs['sex'] = adata_brain.obs['pt'].map(sex_dict)
adata_brain.obs['diagnosis'] = adata_brain.obs['pt'].map(diag_dict)
adata_brain.obs['tissue'] = 'brain'

In [62]:
from collections import Counter
Counter(adata_blood.obs['age'])

Counter({66: 17312,
         36: 30179,
         52: 20003,
         65: 16169,
         72: 12403,
         80: 14866,
         69: 12836})

In [63]:
adata =  ad.concat([adata_brain, adata_blood], join="outer", index_unique="_")
adata.X = adata.layers["counts"].copy()


In [ ]:
np.max(adata.X)


In [ ]:
adata.obs["cell_type"]= adata.obs["cell_type"].astype("str")
adata.obs["cell_type"].fillna("NA", inplace=True)


In [ ]:
adata.obs["pt"] = [pt.replace("-", "_") for pt in adata.obs["pt"]]
adata.obs["cell_type"] = [ct.replace(" ", "_") for ct in adata.obs["cell_type"]]
adata.obs["cell_type"] = [ct.replace("+", "") for ct in adata.obs["cell_type"]]
adata.obs["cell_type"] = [ct.replace("/", "_") for ct in adata.obs["cell_type"]]


In [ ]:
adata.obs["sample"] = [
    f"{rep}_{l}" for rep, l in zip(adata.obs["pt"], adata.obs["tissue"])
]

adata.obs["cell_type"] = adata.obs["cell_type"].astype("category")
adata.obs["sample"] = adata.obs["sample"].astype("category")
adata.obs["label"] = adata.obs["tissue"].astype("category")
adata.obs["replicate"] = adata.obs["pt"].astype("category")
print(pd.unique(adata.obs["cell_type"]),pd.unique(adata.obs["sample"]))

In [ ]:
print(adata.shape)
indices = list(adata.obs_names)
random.shuffle(indices)
indices = np.array_split(np.array(indices), 1)
for i, rep_idx in enumerate(indices):
    print(adata[rep_idx].shape)


In [ ]:
NUM_OF_CELL_PER_DONOR = 10

np.random.seed(42)

def aggregate_and_filter(
    adata,
    cell_identity,
    donor_key="sample",
    condition_key="label",
    cell_identity_key="cell_type",
    obs_to_keep=[],  # which additional metadata to keep, e.g. gender, age, etc.
    replicates_per_patient=1,
):
    # subset adata to the given cell identity
    adata_cell_pop = adata[adata.obs[cell_identity_key] == cell_identity].copy()
    # check which donors to keep according to the number of cells specified with NUM_OF_CELL_PER_DONOR
    size_by_donor = adata_cell_pop.obs.groupby([donor_key]).size()
    donors_to_drop = [
        donor
        for donor in size_by_donor.index
        if size_by_donor[donor] <= NUM_OF_CELL_PER_DONOR
    ]
    if len(donors_to_drop) > 0:
        print("Dropping the following samples:")
        print(donors_to_drop)
    df = pd.DataFrame(columns=[*adata_cell_pop.var_names, *obs_to_keep])

    adata_cell_pop.obs[donor_key] = adata_cell_pop.obs[donor_key].astype("category")
    for i, donor in enumerate(donors := adata_cell_pop.obs[donor_key].cat.categories):
        print(f"\tProcessing donor {i+1} out of {len(donors)}...", end="\r")
        if donor not in donors_to_drop:
            adata_donor = adata_cell_pop[adata_cell_pop.obs[donor_key] == donor]
            # create replicates for each donor
            indices = list(adata_donor.obs_names)
            random.shuffle(indices)
            indices = np.array_split(np.array(indices), replicates_per_patient)
            for i, rep_idx in enumerate(indices):
                adata_replicate = adata_donor[rep_idx]
                # specify how to aggregate: sum gene expression for each gene for each donor and also keep the condition information
                agg_dict = {gene: "sum" for gene in adata_replicate.var_names}
                for obs in obs_to_keep:
                    agg_dict[obs] = "first"
                # create a df with all genes, donor and condition info
                df_donor = pd.DataFrame(adata_replicate.X.A)
                df_donor.index = adata_replicate.obs_names
                df_donor.columns = adata_replicate.var_names
                df_donor = df_donor.join(adata_replicate.obs[obs_to_keep])
                # aggregate
                df_donor = df_donor.groupby(donor_key).agg(agg_dict)
                df_donor[donor_key] = donor
                df.loc[f"donor_{donor}_{i}"] = df_donor.loc[donor]
    print("\n")
    # create AnnData object from the df
    adata_cell_pop = sc.AnnData(
        df[adata_cell_pop.var_names], obs=df.drop(columns=adata_cell_pop.var_names)
    )
    return adata_cell_pop

In [ ]:
%%R
fit_model <- function(adata_){
    # create an edgeR object with counts and grouping factor
    y <- DGEList(assay(adata_, "X"), group = colData(adata_)$label)
    # filter out genes with low counts
    print("Dimensions before subsetting:")
    print(dim(y))
    print("")
    keep <- filterByExpr(y)
    y <- y[keep, , keep.lib.sizes=FALSE]
    print("Dimensions after subsetting:")
    print(dim(y))
    print("")
    # normalize
    y <- calcNormFactors(y)
    # create a vector that is concatentation of condition and cell type that we will later use with contrasts
    group <- paste0(colData(adata_)$label, ".", colData(adata_)$cell_type)
    replicate <- colData(adata_)$replicate
    # create a design matrix: here we have multiple donors so also consider that in the design matrix
    design <- model.matrix(~ 0 + group + replicate)
    # estimate dispersion
    y <- estimateDisp(y, design = design)
    # fit the model
    fit <- glmQLFit(y, design)
    return(list("fit"=fit, "design"=design, "y"=y))
}

In [ ]:
obs_to_keep = ["label", "cell_type", "replicate", "sample"]


In [ ]:
# process first cell type separately...
cell_type = adata.obs["cell_type"].cat.categories[0]
print(
    f'Processing {cell_type} (1 out of {len(adata.obs["cell_type"].cat.categories)})...'
)
adata_pb = aggregate_and_filter(adata, cell_type,  donor_key="sample", condition_key="label", cell_identity_key="cell_type", obs_to_keep=obs_to_keep)
for i, cell_type in enumerate(adata.obs["cell_type"].cat.categories[1:]):
    print(
        f'Processing {cell_type} ({i+2} out of {len(adata.obs["cell_type"].cat.categories)})...'
    )
    adata_cell_type = aggregate_and_filter(adata, cell_type, obs_to_keep=obs_to_keep)
    adata_pb = adata_pb.concatenate(adata_cell_type)

In [ ]:
adata_pb.layers['counts'] = adata_pb.X.copy()
sc.pp.normalize_total(adata_pb, target_sum=1e6)
sc.pp.log1p(adata_pb)
sc.pp.pca(adata_pb)

In [ ]:
adata_pb.obs["lib_size"] = np.sum(adata_pb.layers["counts"], axis=1).astype(int)
# adata_pb.obs["log_lib_size"] = np.log(adata_pb.obs["lib_size"])

In [ ]:
# adata_pb.layers['counts']

In [ ]:
# adata_pb.layers['counts'] = csr_matrix(adata_pb.layers["counts"].astype(np.float32))

# adata_pb.write(
#         project_path
#         / f"adata_pb.h5ad"
#     )
# adata_pb = sc.read(
#         project_path
#         / f"adata_pb.h5ad"
#     )

In [ ]:
# sc.pl.pca(adata_pb, color=adata_pb.obs, ncols=1, size=300)


In [ ]:
# adata_pb.layers['counts'] = csr_matrix(adata_pb.layers["counts"].astype(np.float32))
adata_pb.X = adata_pb.layers['counts'].copy()


In [44]:
adata, adata_pb

(AnnData object with n_obs × n_vars = 166343 × 32542
     obs: 'side', 'pt', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'pct_counts_mt', 'pct_counts_ribo', 'cell_type', 'age', 'sex', 'diagnosis', 'tissue', 'sample', 'label', 'replicate'
     obsm: 'X_pca', 'X_umap'
     layers: 'counts', 'log1p_norm', 'scran_normalization',
 View of AnnData object with n_obs × n_vars = 179 × 27
     obs: 'label', 'cell_type', 'replicate', 'sample', 'batch', 'lib_size'
     uns: 'log1p', 'pca'
     obsm: 'X_pca'
     varm: 'PCs'
     layers: 'counts')

In [45]:
"U2AF1L5" in adata.var_names

True

In [ ]:
# len(corr_pos['blood.symbol'].unique())+ len(corr_neg['blood.symbol'].unique()), len(corr_pos['brain.symbol'].unique())+ len(corr_neg['brain.symbol'].unique())

In [ ]:
# corr_all.sort_values("rho.slimModel")

In [ ]:
# corr_pos.shape, corr_neg.shape

In [ ]:
project_path = Path(
    "/Users/nicolebussola/Desktop/Projects/LBP_MountSinai/LBP_brain_blood_pairs/data/"
)
corr_pos_all = pd.read_csv(project_path / "sig_corr_slimFull_pos.csv")
corr_neg_all = pd.read_csv(project_path / "sig_corr_slimFull_neg.csv")

t = 0.4
corr_pos = corr_pos_all[corr_pos_all['rho.slimModel']>=t].sort_values(by='rho.slimModel')#.iloc[-30:]
corr_neg = corr_neg_all[corr_neg_all['rho.slimModel']<=-t].sort_values(by='rho.slimModel')#.iloc[:30]

corr_all = pd.concat([corr_pos, corr_neg], ignore_index=True, axis=0).reset_index(drop=True)

In [ ]:
genes_blood = list(set(adata.var_names).intersection(set((corr_all['blood.symbol']))))
genes_brain = list(set(adata.var_names).intersection(set((corr_all['brain.symbol']))))
sorted(genes_blood), sorted(genes_brain)

In [64]:
len(np.unique(genes_blood)), len(np.unique(genes_brain)), genes_blood,genes_brain

(14,
 13,
 ['TBC1D3L',
  'LRG1',
  'PAX8-AS1',
  'MANSC1',
  'U2AF1',
  'PFKFB2',
  'NBPF26',
  'SLCO4C1',
  'ALOX5AP',
  'KCNE1',
  'U2AF1L5',
  'TPST1',
  'GATD3B',
  'FKBP5'],
 ['CCDC144A',
  'TBC1D3L',
  'HLA-DQA2',
  'PAX8-AS1',
  'U2AF1',
  'CYLD',
  'U2AF1L5',
  'LINC01597',
  'PDZD8',
  'ZBTB16',
  'GATD3B',
  'FKBP5',
  'OR7A5'])

In [65]:
# project_path = Path(
#     "/Users/nicolebussola/Desktop/Projects/LBP_MountSinai/LBP_brain_blood_pairs/data/TD005881/"
# )
# adata_pb = sc.read(
#         project_path
#         / f"adata_pb.h5ad"
#     )
adata_pb = adata_pb[:, genes_blood+genes_brain]

InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [66]:
adata_pb

View of AnnData object with n_obs × n_vars = 179 × 27
    obs: 'label', 'cell_type', 'replicate', 'sample', 'batch', 'lib_size'
    uns: 'log1p', 'pca'
    obsm: 'X_pca'
    varm: 'PCs'
    layers: 'counts'

In [67]:
adata_mono = adata_pb[adata_pb.obs["cell_type"] == "CD14_Mono"]
adata_mono.var

""
TBC1D3L
LRG1
PAX8-AS1
MANSC1
U2AF1
PFKFB2
NBPF26
SLCO4C1
ALOX5AP
KCNE1


In [68]:
adata_mono.obs

,label,cell_type,replicate,sample,batch,lib_size
donor_PT_280_blood_0-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,blood,CD14_Mono,PT_280,PT_280_blood,0,14178267
donor_PT_282_blood_0-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,blood,CD14_Mono,PT_282,PT_282_blood,0,19875448
donor_PT_285_blood_0-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,blood,CD14_Mono,PT_285,PT_285_blood,0,45385840
donor_PT_286_blood_0-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,blood,CD14_Mono,PT_286,PT_286_blood,0,22830451
donor_PT_289_blood_0-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,blood,CD14_Mono,PT_289,PT_289_blood,0,79512650
donor_PT_290_blood_0-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,blood,CD14_Mono,PT_290,PT_290_blood,0,20255522
donor_PT_292_blood_0-1-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,blood,CD14_Mono,PT_292,PT_292_blood,0,9040664


In [69]:
adata_astro = adata_pb[adata_pb.obs["cell_type"] == "Astrocytes"]
adata_astro.obs

,label,cell_type,replicate,sample,batch,lib_size
donor_PT_280_brain_0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,brain,Astrocytes,PT_280,PT_280_brain,0,3605714
donor_PT_282_brain_0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,brain,Astrocytes,PT_282,PT_282_brain,0,4205024
donor_PT_285_brain_0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,brain,Astrocytes,PT_285,PT_285_brain,0,3155563
donor_PT_286_brain_0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,brain,Astrocytes,PT_286,PT_286_brain,0,4332117
donor_PT_289_brain_0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,brain,Astrocytes,PT_289,PT_289_brain,0,12336884
donor_PT_290_brain_0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,brain,Astrocytes,PT_290,PT_290_brain,0,10098070
donor_PT_292_brain_0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0,brain,Astrocytes,PT_292,PT_292_brain,0,3107904


In [70]:
pd.unique(adata_pb.obs["cell_type"])

array(['Astrocytes', 'B1_B', 'Brain_cells', 'CD14_Mono', 'CD16_Mono',
       'CD4_T_activated', 'CD4_T_naive', 'CD8_T_activated', 'CD8_T_naive',
       'Endothelial', 'Excitatory_neurons', 'Glial', 'HSC',
       'Immature_neurons', 'Inhibitory_neurons', 'MK_E_prog', 'Microglia',
       'NK', 'Naive_CD20_B', 'OPC', 'Other_T', 'Plasma', 'Treg', 'cDC2',
       'nan', 'pDC', 'presynaptic_terminals'], dtype=object)

In [71]:
from scipy.stats import kendalltau

adata_mono = adata_pb[adata_pb.obs["cell_type"] == "CD14_Mono"]
adata_astro = adata_pb[adata_pb.obs["cell_type"] == "Astrocytes"]

print("CD14_Mono-Astrocytes", kendalltau(np.mean(adata_mono.X, axis=0), np.mean(adata_astro.X, axis=0)).correlation)


CD14_Mono-Astrocytes 0.4608695652173913


In [72]:

adata_mono = adata_pb[adata_pb.obs["cell_type"] == "CD16_Mono"]
adata_astro = adata_pb[adata_pb.obs["cell_type"] == "Astrocytes"]

print("CD16_Mono-Astrocytes", kendalltau(np.mean(adata_mono.X, axis=0), np.mean(adata_astro.X, axis=0)).correlation)


CD16_Mono-Astrocytes 0.4898550724637682


In [ ]:
adata_mono = adata_pb[adata_pb.obs["cell_type"] == "NK"]
adata_astro = adata_pb[adata_pb.obs["cell_type"] == "Astrocytes"]

print("NK-Astrocytes", kendalltau(np.mean(adata_mono.X, axis=0), np.mean(adata_astro.X, axis=0)).correlation)


In [ ]:
adata_mono = adata_pb[adata_pb.obs["cell_type"] == "NK"]
adata_astro = adata_pb[adata_pb.obs["cell_type"] == "Astrocytes"]

print("NK-Astrocytes",kendalltau(np.mean(adata_mono.X, axis=0), np.mean(adata_astro.X, axis=0)).correlation)


In [ ]:
adata_mono = adata_pb[adata_pb.obs["cell_type"] == "CD14_Mono"]
adata_astro = adata_pb[adata_pb.obs["cell_type"] == "OPC"]

print("CD14_Mono-OPC",kendalltau(np.mean(adata_mono.X, axis=0), np.mean(adata_astro.X, axis=0)).correlation)

In [ ]:
adata_mono = adata_pb[adata_pb.obs["cell_type"] == "CD16_Mono"]
adata_astro = adata_pb[adata_pb.obs["cell_type"] == "OPC"]

print("CD16_Mono-OPC",kendalltau(np.mean(adata_mono.X, axis=0), np.mean(adata_astro.X, axis=0)).correlation)

In [ ]:
[i for i in pd.unique(adata_pb.obs["cell_type"])]

In [ ]:
adata_mono = adata_pb[adata_pb.obs["cell_type"] == "Naive_CD20_B"]
adata_astro = adata_pb[adata_pb.obs["cell_type"] == "OPC"]

print("Naive_CD20_B-OPC",kendalltau(np.mean(adata_mono.X, axis=0), np.mean(adata_astro.X, axis=0)).correlation)

In [ ]:
adata_mono = adata_pb[adata_pb.obs["cell_type"] == "B1_B"]
adata_astro = adata_pb[adata_pb.obs["cell_type"] == "OPC"]

print("B1_B",kendalltau(np.mean(adata_mono.X, axis=0), np.mean(adata_astro.X, axis=0)).correlation)

In [ ]:
adata_mono = adata_pb[adata_pb.obs["cell_type"] == "NK"]
adata_astro = adata_pb[adata_pb.obs["cell_type"] == "Excitatory_neurons"]

print("NK",kendalltau(np.mean(adata_mono.X, axis=0), np.mean(adata_astro.X, axis=0)).correlation)

In [ ]:
adata_mono = adata_pb[adata_pb.obs["cell_type"] == "NK"]
adata_astro = adata_pb[adata_pb.obs["cell_type"] == "Astrocytes"]

print("NK",kendalltau(np.mean(adata_mono.X, axis=0), np.mean(adata_astro.X, axis=0)).correlation)

In [ ]:
adata_mono.obs['lib_size']=adata_mono.obs['lib_size'].astype(int)
adata_mono.obs['lib_size']

In [ ]:
%%time
%%R -i adata_mono
library(edgeR)
outs <-fit_model(adata_mono)

In [ ]:
%%R
fit <- outs$fit
y <- outs$y

In [ ]:
%%R
plotMDS(y, col=ifelse(y$samples$group == "stim", "red", "blue"))

In [ ]:
%%R
plotBCV(y)

In [ ]:
%%R
colnames(y$design)


In [ ]:
%%R -o tt
myContrast <- makeContrasts("groupL.CD14_Mono-groupR.CD14_Mono", levels = y$design)
qlf <- glmQLFTest(fit, contrast=myContrast)
# get all of the DE genes and calculate Benjamini-Hochberg adjusted FDR
tt <- topTags(qlf, n = Inf)
tt <- tt$table

In [ ]:
tt[:5]


In [ ]:
adata_pb.obs['lib_size']=adata_pb.obs['lib_size'].astype(int)


In [ ]:
%%time
%%R -i adata_pb
outs <-fit_model(adata_pb)

In [ ]:
%%R
fit <- outs$fit
y <- outs$y

In [ ]:
%%R -i adata_pb -o de_per_cell_type
de_per_cell_type <- list()
for (cell_type in unique(colData(adata_pb)$cell_type)) {
    print(cell_type)
    # create contrast for this cell type
    myContrast <- makeContrasts(paste0("groupR.", cell_type, "-groupL.", cell_type), levels = y$design)
    # perform QLF test
    qlf <- glmQLFTest(fit, contrast=myContrast)
    # get all of the DE genes and calculate Benjamini-Hochberg adjusted FDR
    tt <- topTags(qlf, n = Inf)
    # save in the list with the results for all the cell types
    de_per_cell_type[[cell_type]] <- tt$table
}

In [ ]:
# get cell types that we ran the analysis for
cell_types = de_per_cell_type.keys()
# add the table to .uns for each cell type
for cell_type in cell_types:
    df = de_per_cell_type[cell_type]
    df["gene_symbol"] = df.index
    df["cell_type"] = cell_type
    sc_toolbox.tools.de_res_to_anndata(
        adata,
        df,
        groupby="cell_type",
        score_col="logCPM",
        pval_col="PValue",
        pval_adj_col="FDR",
        lfc_col="logFC",
        key_added="edgeR_" + cell_type,
    )
    df.to_csv(f"de_edgeR_{cell_type}.csv")

In [ ]:
sc.get.rank_genes_groups_df(adata, group="CD14_Mono", key="edgeR_CD14_Mono")[
    :5

]

In [ ]:
FDR = 0.01
LOG_FOLD_CHANGE = 1.5



def plot_heatmap(adata, group_key, group_name="cell_type", groupby="label"):
    cell_type = "_".join(group_key.split("_")[1:])
    res = sc.get.rank_genes_groups_df(adata, group=cell_type, key=group_key)
    res.index = res["names"].values
    res = res[
        (res["pvals_adj"] < FDR) & (abs(res["logfoldchanges"]) > LOG_FOLD_CHANGE)
    ].sort_values(by=["logfoldchanges"])
    print(f"Plotting {len(res)} genes...")
    markers = list(res.index)
    sc.pl.heatmap(
        adata[adata.obs[group_name] == cell_type].copy(),
        markers,
        groupby=groupby,
        swap_axes=True,
    )

def volcano_plot(adata, group_key, group_name="cell_type", groupby="label", title=None):
    cell_type = "_".join(group_key.split("_")[1:])
    result = sc.get.rank_genes_groups_df(adata, group=cell_type, key=group_key).copy()
    result["-logQ"] = -np.log(result["pvals"].astype("float"))
    lowqval_de = result.loc[abs(result["logfoldchanges"]) > LOG_FOLD_CHANGE]
    other_de = result.loc[abs(result["logfoldchanges"]) <= LOG_FOLD_CHANGE]

    fig, ax = plt.subplots()
    sns.regplot(
        x=other_de["logfoldchanges"],
        y=other_de["-logQ"],
        fit_reg=False,
        scatter_kws={"s": 6},
    )
    sns.regplot(
        x=lowqval_de["logfoldchanges"],
        y=lowqval_de["-logQ"],
        fit_reg=False,
        scatter_kws={"s": 6},
    )
    ax.set_xlabel("log2 FC")
    ax.set_ylabel("-log Q-value")

    if title is None:
        title = group_key.replace("_", " ")
    plt.title(title)
    plt.show()

In [ ]:
plot_heatmap(adata, "edgeR_CD14_Mono")


In [ ]:
volcano_plot(adata, "edgeR_CD14_Mono")


In [ ]:
from bioinfokit import analys, visuz


In [ ]:
df = sc.get.rank_genes_groups_df(adata, group="CD14_Mono", key="edgeR_CD14_Mono")
df["pvals"] = df["pvals"].astype("float")

df["-logQ"] = -np.log(df["pvals"])
visuz.GeneExpression.volcano(df=df, lfc='logfoldchanges', pv='pvals',show=True, color=( "#DB141C", "#D4D4D4","#0444B4"), geneid="names", genenames='deg', lfc_thr= [LOG_FOLD_CHANGE, LOG_FOLD_CHANGE],gstyle=2, sign_line=True , dim=(5,5))


In [ ]:
result = sc.get.rank_genes_groups_df(adata, group="CD14_Mono", key="edgeR_CD14_Mono").copy()
result["-logQ"] = -np.log(result["pvals"].astype("float"))
result

In [ ]:
adata.uns['log1p']['base'] = None

In [ ]:
?sc.get.rank_genes_groups_df

In [ ]:
?dc.plot_volcano

In [ ]:
logFCs, pvals = dc.get_contrast(
    adata,
    group_col="cell_type",
    condition_col="side",
    condition="L",
    reference="R",
    method="wilcoxon",
)

In [ ]:
import decoupler as dc
dc.plot_volcano(logFCs, pvals, "CD14_Mono", top=5, sign_thr=0.05, lFCs_thr=1.5)


In [ ]:
np.log(0.582187)

In [ ]:
import pertpy as pt

In [ ]:
sccoda_model = pt.tl.Sccoda()

In [ ]:
adata

In [ ]:
sccoda_data = sccoda_model.load(
    adata,
    type="cell_level",
    generate_sample_level=True,
    cell_type_identifier="cell_type",
    sample_identifier="sample",
    covariate_obs=["side"],
)
sccoda_data

In [ ]:
pt.pl.coda.boxplots(
    sccoda_data,
    modality_key="coda",
    feature_name="side",
    figsize=(12, 5),
    add_dots=True,
    args_swarmplot={"palette": ["red"]},
)
plt.show()

In [ ]:
pt.pl.coda.stacked_barplot(
    sccoda_data, modality_key="coda", feature_name="side", figsize=(4, 2)
)
plt.show()

In [ ]:
sccoda_data = sccoda_model.prepare(
    sccoda_data,
    modality_key="coda",
    formula="side",
    reference_cell_type="automatic",
)
sccoda_model.run_nuts(sccoda_data, modality_key="coda", rng_key=1234)

In [ ]:
sccoda_data["coda"].varm

In [ ]:
sccoda_data["coda"].varm['effect_df_side[T.R]']


In [ ]:
sccoda_model.set_fdr(sccoda_data, 0.2)


In [ ]:
sccoda_model.credible_effects(sccoda_data, modality_key="coda")


In [ ]:
pt.pl.coda.effects_barplot(sccoda_data, "coda", "side")
plt.show()